# CoT-Injection Monitoring — Colab Launcher
Parametrised harness. Set your keys and config in **Cell 2**, then run top-to-bottom.

| GPU needed | CPU fine |
|---|---|
| Cell 4 (subject), Cell 5 (clean baseline) | 1, 2, 3, 6, 7 |

> **Tip:** Use *Runtime → Run all* for a full run, or run cells individually to resume.

In [ ]:
# Cell 1 — Clone + install
# Replace YOUR_REPO_URL with the output of: git remote get-url origin
REPO_URL = "https://github.com/YOUR_REPO_URL"
REPO_DIR = "/content/cot-injection-monitoring"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q uv
!uv sync --extra gpu --extra openrouter --extra dev
print('Done')

In [ ]:
# Cell 2 — Keys + run config  ← EDIT THIS CELL
import os

os.environ['OPENROUTER_API_KEY'] = 'sk-or-v1-...'   # required
os.environ['HF_TOKEN']           = 'hf_...'          # required for gated models

# Which experiment to run — pick one of the files in experiments/ or set your own.
CONFIG = 'experiments/example_h2.yaml'

# Optional: shard subject inference across multiple sessions (0-based).
SHARD = 0
TOTAL_SHARDS = 1

print(f'Config: {CONFIG}  shard {SHARD}/{TOTAL_SHARDS}')

In [ ]:
# Cell 3 — Mount Drive + workspace + GPU check
from src.infra.paths import resolve_root
paths = resolve_root()   # mounts Drive on Colab; sets COTIM_ROOT
print('Workspace:', paths.root)

import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 3b — Dry-run: estimate cost + matrix size (zero spend, no GPU)
!python -m src.pipeline.stages --config {CONFIG} --dry-run

In [ ]:
# Cell 4 — [GPU] Baselines + prefills + subject inference
# Baselines/prefills: CPU+API. Subject: GPU. All write cache immediately.
SHARD_FLAGS = f'--shard {SHARD} --total-shards {TOTAL_SHARDS}' if TOTAL_SHARDS > 1 else ''
!bash scripts/run_datagen.sh {CONFIG} all_datagen {SHARD_FLAGS}

In [ ]:
# Cell 5 — [GPU] Clean (no-injection) baseline run  [needed for causal EEMR]
# Already included in all_datagen above; run separately only if you skipped it.
# !bash scripts/run_datagen.sh {CONFIG} clean_baseline {SHARD_FLAGS}

In [ ]:
# Cell 6 — [API] Judges + metrics + plots  (no GPU needed)
!bash scripts/run_pipeline.sh {CONFIG}

In [ ]:
# Cell 7 — View results
import json
from pathlib import Path
import yaml

cfg = yaml.safe_load(open(CONFIG))
run_name = cfg.get('run', {}).get('run_name', 'run')
metrics_path = Path(f'results/{run_name}/metrics.json')

if not metrics_path.exists():
    print('metrics.json not found — run Cell 6 first.')
else:
    summary = json.loads(metrics_path.read_text())
    print(f"Enriched results : {summary['n_enriched']}")
    print(f"Condition cells  : {len(summary['conditions'])}")
    print(f"CAS entries      : {len(summary['cas'])}")
    print(f"Dissociation rate: {summary['dissociation']['rate']:.3f}")

    # Show monitor-capability table
    cap_path = Path(f'results/{run_name}/monitor_capability.json')
    if cap_path.exists():
        from src.pipeline.stages import _print_capability_table
        _print_capability_table(json.loads(cap_path.read_text()))